# 02 — Preprocessing

Normalises columns, imputes missing values, engineers financial ratios, and saves
 for use in .

## 1. Imports & configuration

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "data"))
OUT_PATH = DATA_DIR / "processed_data.csv"

# ── Thresholds (edit here — do not scatter through the notebook) ─────────────
INVESTMENT_AMOUNT  = 2_000_000  #  M assumed investment
ROI_THRESHOLD      = 0.05
OP_MARGIN_THRESHOLD = 0.02
DTE_THRESHOLD      = 1.5        # Debt-to-equity
CR_THRESHOLD       = 1.5        # Current ratio
DAR_THRESHOLD      = 80         # Days in accounts receivable
BOR_THRESHOLD      = 0.80       # Bed occupancy rate

print(f"Output will be saved to: {OUT_PATH}")

## 2. Load raw data

In [ ]:
CSV_FILES = [f"{yr}CostReport.csv" for yr in range(2015, 2022)]

dfs = []
for fname in CSV_FILES:
    fpath = DATA_DIR / fname
    df = pd.read_csv(fpath, low_memory=False)
    df["year"] = fname[:4]
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(dfs)} files — merged shape: {raw.shape}")

## 3. Column normalisation

Strip whitespace, lowercase, remove special characters. One canonical definition — no duplicate functions.

In [ ]:
def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase, strip whitespace, remove all non-alphanumeric characters."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "", regex=False)
        .str.replace(r"[^a-z0-9]", "", regex=True)
    )
    return df

data = normalize_column_names(raw)
print("Columns normalised. Sample:", data.columns.tolist()[:10])

## 4. Drop uninformative columns

In [ ]:
# Drop date columns that carry no modelling value
date_cols = [c for c in data.columns if "fiscalyear" in c or ("date" in c and "begin" in c) or ("date" in c and "end" in c)]
print("Dropping:", date_cols)
data = data.drop(columns=date_cols, errors="ignore")

## 5. Feature engineering

All ratio features are derived here — before imputation — so that imputation
operates on the complete final feature set.

In [ ]:
data["ROI"]                     = data["netincome"] / INVESTMENT_AMOUNT
data["OperatingMargin"]          = data["netincome"] / data["grossrevenue"]
data["Debttoequityratio"]        = data["totalliabilities"] / data["totalfundbalances"]
data["CurrentRatio"]             = data["totalcurrentassets"] / data["totalcurrentliabilities"]
data["DaysinAccountsReceivable"] = (data["accountsreceivable"] / data["netpatientrevenue"]) * 365
data["BedOccupancyRate"]         = (data["totaldaystotal"] / (data["numberofbeds"] * 365)) * 100

print("Engineered features — descriptive stats:")
ratio_cols = ["ROI", "OperatingMargin", "Debttoequityratio",
              "CurrentRatio", "DaysinAccountsReceivable", "BedOccupancyRate"]
print(data[ratio_cols].describe().T[["count", "mean", "std", "min", "max"]])

## 6. Target variable: 

In [ ]:
def calculate_investment_choice(df: pd.DataFrame) -> pd.DataFrame:
    """Binary label: 1 = invest, 0 = pass. Uses module-level thresholds."""
    df = df.copy()
    c1 = (df["ROI"] > ROI_THRESHOLD) | (df["OperatingMargin"] > OP_MARGIN_THRESHOLD)
    c2 = (df["Debttoequityratio"] < DTE_THRESHOLD) | (df["CurrentRatio"] > CR_THRESHOLD)
    c3 = (df["DaysinAccountsReceivable"] < DAR_THRESHOLD) | (df["BedOccupancyRate"] > BOR_THRESHOLD)
    df["Investment_Choice"] = np.where(c1 & c2 & c3, 1, 0)
    return df

data = calculate_investment_choice(data)
print("Target distribution:")
print(data["Investment_Choice"].value_counts(normalize=True).rename("proportion").round(3))

## 7. Train/test split — BEFORE imputation

**Important:** the scaler and imputer are fit only on training data to prevent data leakage.
Imputed values for the test and validation sets are derived solely from training statistics.

In [ ]:
from sklearn.model_selection import train_test_split

num_cols = data.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = data.select_dtypes(include=["object", "category", "str"]).columns.tolist()

# Remove target from feature lists
num_cols = [c for c in num_cols if c != "Investment_Choice"]

X = data[num_cols + cat_cols]
y = data["Investment_Choice"]

# 75% train | 15% test | 10% val
X_train, X_rem, y_train, y_rem = train_test_split(X, y, test_size=0.25, random_state=42)
X_test,  X_val, y_test,  y_val = train_test_split(X_rem, y_rem, test_size=0.40, random_state=42)

print(f"Train: {len(X_train)} | Test: {len(X_test)} | Val: {len(X_val)}")

## 8. Missing value imputation

Median imputation is used for numerical columns (robust to the financial outliers seen in the EDA).
Mode imputation is used for categorical columns.
Both imputers are fit on training data only.

In [ ]:
# Numerical: median (robust to skewed financial data)
num_imputer = SimpleImputer(strategy="median")
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols]  = num_imputer.transform(X_test[num_cols])
X_val[num_cols]   = num_imputer.transform(X_val[num_cols])

# Categorical: most_frequent
cat_imputer = SimpleImputer(strategy="most_frequent")
if cat_cols:
    X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
    X_test[cat_cols]  = cat_imputer.transform(X_test[cat_cols])
    X_val[cat_cols]   = cat_imputer.transform(X_val[cat_cols])

# Verify
assert X_train[num_cols].isnull().sum().sum() == 0, "Numerical NaNs remain in train"
assert X_test[num_cols].isnull().sum().sum() == 0,  "Numerical NaNs remain in test"
print("Imputation complete — no remaining NaNs in numerical columns.")

## 9. Outlier inspection (IQR clipping)

Engineered ratio columns can produce extreme values from division by near-zero denominators.
We clip at 1st/99th percentiles to limit the effect on model training.

In [ ]:
CLIP_COLS = ["OperatingMargin", "Debttoequityratio", "CurrentRatio", "DaysinAccountsReceivable", "BedOccupancyRate"]

for col in CLIP_COLS:
    if col in X_train.columns:
        lo = X_train[col].quantile(0.01)
        hi = X_train[col].quantile(0.99)
        for split in [X_train, X_test, X_val]:
            split[col] = split[col].clip(lo, hi)
        print(f"{col}: clipped to [{lo:.3f}, {hi:.3f}]")

## 10. Save processed splits

In [ ]:
# Reassemble train/test/val with target and save a single processed file
# (split indices are preserved via random_state=42 and can be reproduced in 03_modeling)
train_out = X_train.copy(); train_out["Investment_Choice"] = y_train.values; train_out["_split"] = "train"
test_out  = X_test.copy();  test_out["Investment_Choice"]  = y_test.values;  test_out["_split"]  = "test"
val_out   = X_val.copy();   val_out["Investment_Choice"]   = y_val.values;   val_out["_split"]   = "val"

processed = pd.concat([train_out, test_out, val_out], ignore_index=True)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
processed.to_csv(OUT_PATH, index=False)
print(f"Saved {processed.shape} rows to {OUT_PATH}")
print(processed["_split"].value_counts())